# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepikachebrolu/flyrank-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*



### Project: CTR / Engagement Opportunity Scoring

### Objective

This notebook converts observable content-performance metrics into a ranked queue of potential content actions.

The queue is designed to help a human reviewer identify pages that may deserve investigation. It does not automatically change content, guarantee performance improvements, or establish that a particular intervention will cause an increase in CTR.

### Approach

The Week 6 validation did not show that the Random Forest consistently outperformed the median baseline. Therefore, this playbook does not treat model predictions as reliable estimates of future performance.

Instead, it uses transparent, rule-based prioritization based on observed signals such as impressions, CTR, average search position, engagement, and content freshness.

Each recommendation includes:
- A priority score and rank.
- An action archetype.
- A reason code explaining why the item was selected.
- A suggested next step for a human reviewer.
- The observed metrics supporting the recommendation.

The rules are prioritization heuristics, not a validated estimate of the causal impact of content changes.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

current = Path.cwd().resolve()

ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Could not find data/raw/content_refresh_anonymized.csv. "
        "Run this notebook from inside the flyrank-ml repository."
    )

DATA_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
OUTPUT_DIR = ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

required_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "sessions_90d",
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
print("Repository root:", ROOT)


numeric_columns = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "sessions_90d",
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")


work = df.dropna(
    subset=["content_id", "impressions_90d", "ctr"]
).copy()


work = work[
    np.isfinite(work["impressions_90d"])
    & np.isfinite(work["ctr"])
    & (work["impressions_90d"] >= 0)
].copy()

if work.empty:
    raise ValueError("No usable rows remain after data validation.")

visible = work[work["impressions_90d"] > 0].copy()

if visible.empty:
    raise ValueError(
        "No rows have positive impressions; cannot build the queue."
    )

high_impressions_threshold = visible["impressions_90d"].quantile(0.75)
low_ctr_threshold = visible["ctr"].quantile(0.25)


position_mask = (
    work["avg_position"].notna()
    & (work["avg_position"] >= 4)
    & (work["avg_position"] <= 20)
)

stale_threshold_days = 180

work["high_impressions"] = (
    work["impressions_90d"] >= high_impressions_threshold
)

work["low_ctr"] = (
    work["ctr"] <= low_ctr_threshold
)

work["stale_content"] = (
    work["days_since_last_update"].notna()
    & (work["days_since_last_update"] >= stale_threshold_days)
)

work["position_opportunity"] = position_mask


engaged = work[
    work["sessions_90d"].notna()
    & (work["sessions_90d"] > 0)
    & work["engagement_rate"].notna()
]

if len(engaged) > 0:
    low_engagement_threshold = engaged["engagement_rate"].quantile(0.25)
else:
    low_engagement_threshold = np.nan

work["low_engagement"] = (
    work["sessions_90d"].notna()
    & (work["sessions_90d"] > 0)
    & work["engagement_rate"].notna()
    & (work["engagement_rate"] <= low_engagement_threshold)
)


def assign_action(row):
    """
    Return an action archetype, reason code, and review instruction.

    Priority order determines which single primary action is assigned
    when a record meets multiple rules. All observed signals remain
    available in the exported queue for reviewer context.
    """

    high_impressions = bool(row["high_impressions"])
    low_ctr = bool(row["low_ctr"])
    stale = bool(row["stale_content"])
    position_opportunity = bool(row["position_opportunity"])
    low_engagement = bool(row["low_engagement"])

    # Priority 1: visible content with relatively low CTR
    if high_impressions and low_ctr:
        return pd.Series({
            "action_archetype": "CTR_REVIEW",
            "reason_code": "HIGH_IMPRESSIONS_LOW_CTR",
            "recommended_action": (
                "Review title, snippet, search intent, and SERP context."
            ),
            "review_reason": (
                "High relative visibility with low relative CTR; "
                "investigate before changing the page."
            ),
        })

    # Priority 2: older content with meaningful visibility
    if high_impressions and stale:
        return pd.Series({
            "action_archetype": "FRESHNESS_REVIEW",
            "reason_code": "STALE_HIGH_VISIBILITY",
            "recommended_action": (
                "Check factual accuracy, outdated examples, links, "
                "and whether the search intent has changed."
            ),
            "review_reason": (
                "Content has not been updated recently and has "
                "relatively high impressions."
            ),
        })

    # Priority 3: visible content ranking outside the top positions
    if high_impressions and position_opportunity:
        return pd.Series({
            "action_archetype": "RANKING_REVIEW",
            "reason_code": "VISIBLE_POSITION_4_TO_20",
            "recommended_action": (
                "Inspect query relevance, page usefulness, internal "
                "links, and competing search results."
            ),
            "review_reason": (
                "The page has relatively high impressions and an "
                "average position between 4 and 20."
            ),
        })

    # Priority 4: observed engagement may merit investigation
    if low_engagement:
        return pd.Series({
            "action_archetype": "ENGAGEMENT_REVIEW",
            "reason_code": "LOW_RELATIVE_ENGAGEMENT",
            "recommended_action": (
                "Review landing-page relevance, readability, content "
                "structure, and the engagement measurement definition."
            ),
            "review_reason": (
                "Observed engagement is in the lower quartile among "
                "records with sessions."
            ),
        })

    # Priority 5: freshness review without the visibility condition
    if stale:
        return pd.Series({
            "action_archetype": "FRESHNESS_REVIEW",
            "reason_code": "STALE_CONTENT_REVIEW",
            "recommended_action": (
                "Check whether the content is outdated or needs "
                "maintenance; do not refresh solely because of age."
            ),
            "review_reason": (
                "The last update was at least 180 days ago."
            ),
        })

    # No rule matched: do not force an action
    return pd.Series({
        "action_archetype": "MONITOR",
        "reason_code": "NO_PRIORITY_RULE_MATCH",
        "recommended_action": (
            "Monitor performance and review only if business context "
            "or new evidence indicates a need."
        ),
        "review_reason": (
            "The record did not meet the current prioritization rules."
        ),
    })


action_fields = work.apply(assign_action, axis=1)
work = pd.concat([work, action_fields], axis=1)



# The score is a rule-based review-priority index, not a probability,
# predicted CTR, expected uplift, or financial value.

priority_weights = {
    "HIGH_IMPRESSIONS_LOW_CTR": 5,
    "STALE_HIGH_VISIBILITY": 4,
    "VISIBLE_POSITION_4_TO_20": 3,
    "LOW_RELATIVE_ENGAGEMENT": 2,
    "STALE_CONTENT_REVIEW": 1,
    "NO_PRIORITY_RULE_MATCH": 0,
}

work["priority_score"] = (
    work["reason_code"].map(priority_weights).fillna(0).astype(int)
)

work = work.sort_values(
    by=[
        "priority_score",
        "impressions_90d",
        "days_since_last_update",
    ],
    ascending=[False, False, False],
    na_position="last",
).reset_index(drop=True)

work["priority_rank"] = np.arange(1, len(work) + 1)



queue_columns = [
    "priority_rank",
    "priority_score",
    "content_id",
    "client_id",
    "action_archetype",
    "reason_code",
    "recommended_action",
    "review_reason",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "sessions_90d",
    "engagement_rate",
]

queue = work[queue_columns].copy()

queue_path = OUTPUT_DIR / "w07_ranked_action_queue.csv"
queue.to_csv(queue_path, index=False)

print("\nRanked queue created.")
print("Queue rows:", len(queue))
print("Exported to:", queue_path)

print("\nTop 15 recommended review items:")
display(queue.head(15))

print("\nAction counts:")
display(
    queue["action_archetype"]
    .value_counts()
    .rename_axis("action_archetype")
    .to_frame("count")
)

print("\nReason-code counts:")
display(
    queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .to_frame("count")
)


action_counts = (
    queue["action_archetype"]
    .value_counts()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 5))

action_counts.plot(kind="barh", ax=ax)

ax.set_title("Content Action Queue by Primary Action Archetype")
ax.set_xlabel("Number of records")
ax.set_ylabel("Action archetype")

fig.tight_layout()

figure_path = OUTPUT_DIR / "w07_action_priority_distribution.png"
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Figure exported to:", figure_path)

summary = {
    "assignment": "ML-10 Content Action Playbook",
    "project_lane": "CTR / Engagement Opportunity Scoring",
    "dataset": str(DATA_PATH.relative_to(ROOT)),
    "input_rows": int(len(df)),
    "queue_rows": int(len(queue)),
    "unique_clients": int(work["client_id"].nunique()),
    "ranking_method": (
        "Transparent rule-based priority index; not a model prediction."
    ),
    "thresholds": {
        "high_impressions_quantile": 0.75,
        "high_impressions_threshold": float(high_impressions_threshold),
        "low_ctr_quantile": 0.25,
        "low_ctr_threshold": float(low_ctr_threshold),
        "stale_content_days": stale_threshold_days,
        "low_engagement_quantile": 0.25,
        "low_engagement_threshold": (
            float(low_engagement_threshold)
            if pd.notna(low_engagement_threshold)
            else None
        ),
    },
    "priority_weights": priority_weights,
    "action_counts": {
        str(key): int(value)
        for key, value in queue["action_archetype"].value_counts().items()
    },
    "reason_code_counts": {
        str(key): int(value)
        for key, value in queue["reason_code"].value_counts().items()
    },
    "intended_use": (
        "Human-reviewed content maintenance and investigation queue."
    ),
    "limits": [
        "Priority scores are heuristic ranks, not probabilities.",
        "The queue does not predict CTR uplift or causal impact.",
        "Relative thresholds depend on this dataset's distribution.",
        "Missing or noisy metrics may affect prioritization.",
        "Content age alone does not prove content decay.",
        "No content changes are automated.",
    ],
    "human_review_required": True,
    "production_ready": False,
    "monitoring_triggers": [
        "Review priority thresholds when the data distribution changes.",
        "Investigate missingness or schema changes.",
        "Compare post-action outcomes with a documented baseline.",
        "Revalidate the model before considering model-based ranking.",
    ],
    "exports": {
        "ranked_queue": str(queue_path.relative_to(ROOT)),
        "action_distribution_figure": str(figure_path.relative_to(ROOT)),
    },
}

summary_path = OUTPUT_DIR / "w07_action_playbook_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nSummary exported to:", summary_path)
print("\nSection 1 complete.")

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.